# TP4 — Couche Bronze avec validation Pydantic

Ce notebook implémente le modèle relationnel complet avec **validation déclarative Pydantic** :
- `TelemetryRow` et `IncidentRow` définissent les règles métier par colonne
- Chaque ligne du CSV est validée individuellement — les rejets sont tracés dans `parse_ok_reason`
- Les violations sont centralisées dans `data_quality_issue`
- Les batches sont tracés dans `ingestion_batch`
- Les opérateurs sont pseudonymisés dans `operator`

## 0. Connexion

In [1]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
import pandas as pd
import hashlib
from datetime import datetime, timezone
from pydantic import BaseModel, field_validator, model_validator, ValidationError
from typing import Optional

db_url = URL.create(
    drivername='postgresql+psycopg2',
    username='indusense_user',
    password='ThEP@ssW0rd',
    host='localhost',
    port=5432,
    database='indusense_db',
)
engine = create_engine(db_url)
with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar())


PostgreSQL 15.1 (Debian 15.1-1.pgdg110+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 10.2.1-6) 10.2.1 20210110, 64-bit


## 1. Chargement des CSV

In [2]:
df_tel_raw = pd.read_csv('telemetry.csv', dtype=str)
df_inc_raw = pd.read_csv('releves_incidents.csv', dtype=str, encoding='utf-8')
print(f'Telemetry brut : {df_tel_raw.shape}')
print(f'Incidents brut : {df_inc_raw.shape}')


Telemetry brut : (135626, 7)
Incidents brut : (1245, 18)


## 2. Modèles de validation Pydantic

Chaque modèle définit les **règles métier au niveau de la ligne** :
- Les types (`float`, `int`) sont vérifiés automatiquement par Pydantic
- Les `@field_validator` ajoutent les règles métier (plages, flags binaires)
- `Optional[float]` permet d'accepter les valeurs manquantes sans rejeter la ligne

> La règle de doublon horaire est **cross-lignes** — elle sera traitée en section 3b par pandas.


In [3]:
TYPE_COLS = [
    'type_surchauffe', 'type_baisse_pression', 'type_vibration',
    'type_bruit_mecanique', 'type_surconsommation', 'type_blocage_mecanique',
    'type_alarme_capteur', 'type_arret_urgence', 'type_defaut_qualite',
]


class TelemetryRow(BaseModel):
    machine_id        : str
    timestamp         : str
    temperature_c     : Optional[float] = None
    pressure_bar      : Optional[float] = None
    voltage_mean_v    : Optional[float] = None
    rotation_mean_rpm : Optional[float] = None
    pieces_produced   : Optional[int]   = None

    model_config = {'coerce_numbers_to_str': False, 'str_strip_whitespace': True}

    @field_validator('machine_id', 'timestamp')
    @classmethod
    def not_empty(cls, v):
        if not v or not v.strip():
            raise ValueError('champ obligatoire vide')
        return v

    @field_validator('temperature_c')
    @classmethod
    def temp_plausible(cls, v):
        if v is not None and not (-50 <= v <= 500):
            raise ValueError(f'temperature hors plage [-50, 500] : {v}')
        return v

    @field_validator('pressure_bar')
    @classmethod
    def pressure_plausible(cls, v):
        if v is not None and not (0 <= v <= 1000):
            raise ValueError(f'pression hors plage [0, 1000] : {v}')
        return v

    @field_validator('pieces_produced')
    @classmethod
    def pieces_positive(cls, v):
        if v is not None and v < 0:
            raise ValueError(f'pieces_produced negatif : {v}')
        return v


class IncidentRow(BaseModel):
    incident_id    : str
    date           : str
    time           : str
    operator_name  : Optional[str] = None
    machine_id     : str
    severity       : int
    operator_badge : Optional[str] = None
    comment        : Optional[str] = None
    shift          : Optional[str] = None
    type_surchauffe        : int
    type_baisse_pression   : int
    type_vibration         : int
    type_bruit_mecanique   : int
    type_surconsommation   : int
    type_blocage_mecanique : int
    type_alarme_capteur    : int
    type_arret_urgence     : int
    type_defaut_qualite    : int

    model_config = {'str_strip_whitespace': True}

    @field_validator('severity')
    @classmethod
    def severity_range(cls, v):
        if not (1 <= v <= 5):
            raise ValueError(f'severity hors [1-5] : {v}')
        return v

    @field_validator(*TYPE_COLS)
    @classmethod
    def binary_flag(cls, v):
        if v not in (0, 1):
            raise ValueError(f'flag binaire attendu (0 ou 1), recu : {v}')
        return v


print('Modèles Pydantic définis : TelemetryRow, IncidentRow')

Modèles Pydantic définis : TelemetryRow, IncidentRow


## 3. Validation ligne par ligne

### 3a. Télémétrie — validation Pydantic

Pour chaque ligne du CSV, on tente `TelemetryRow(**row)`.  
En cas d'échec, `ValidationError` fournit le détail des erreurs.


In [4]:
def parse_ok_from_error(e: ValidationError) -> str:
    parts = []
    for err in e.errors():
        field = '.'.join(str(x) for x in err['loc'])
        parts.append(f"{field} : {err['msg']}")
    return ' | '.join(parts)


tel_valid, tel_invalid = [], []

for row in df_tel_raw.to_dict('records'):
    # Convertit les chaînes vides en None pour Pydantic
    cleaned = {k: (None if v == '' or (isinstance(v, float) and pd.isna(v)) else v)
               for k, v in row.items()}
    try:
        validated = TelemetryRow(**cleaned)
        tel_valid.append({**validated.model_dump(), 'parse_ok': True, 'parse_ok_reason': ''})
    except ValidationError as e:
        tel_invalid.append({**cleaned, 'parse_ok': False,
                             'parse_ok_reason': parse_ok_from_error(e)})

df_tel = pd.DataFrame(tel_valid + tel_invalid)
print(f'parse_ok=True  : {len(tel_valid):,}')
print(f'parse_ok=False : {len(tel_invalid):,}')
if tel_invalid:
    display(pd.DataFrame(tel_invalid)[['machine_id','timestamp','parse_ok_reason']].head(3))


parse_ok=True  : 135,626
parse_ok=False : 0


### 3b. Télémétrie — règle cross-lignes : doublon horaire

Cette règle ne peut pas être exprimée dans Pydantic (elle nécessite de connaître toutes les lignes).  
On la détecte via pandas après validation individuelle.


In [5]:
counts = df_tel[df_tel['parse_ok']].groupby(
    ['machine_id', 'timestamp']
)['machine_id'].transform('count')

mask_dup = counts > 1
df_tel.loc[mask_dup.index[mask_dup], 'parse_ok'] = False
df_tel.loc[mask_dup.index[mask_dup], 'parse_ok_reason'] = (
    'DOUBLON_HORAIRE : ' + counts[mask_dup].astype(str)
    + ' releves pour ' + df_tel.loc[mask_dup.index[mask_dup], 'machine_id']
    + ' a ' + df_tel.loc[mask_dup.index[mask_dup], 'timestamp']
)

print(f'parse_ok=True  final : {df_tel["parse_ok"].sum():,}')
print(f'parse_ok=False final : {(~df_tel["parse_ok"]).sum():,}')


parse_ok=True  final : 132,940
parse_ok=False final : 2,686


### 3c. Incidents — validation Pydantic

In [6]:
inc_valid, inc_invalid = [], []

for row in df_inc_raw.to_dict('records'):
    cleaned = {k: (None if v == '' or (isinstance(v, float) and pd.isna(v)) else v)
               for k, v in row.items()}
    try:
        validated = IncidentRow(**cleaned)
        inc_valid.append({**validated.model_dump(), 'parse_ok': True, 'parse_ok_reason': ''})
    except ValidationError as e:
        inc_invalid.append({**cleaned, 'parse_ok': False,
                             'parse_ok_reason': parse_ok_from_error(e)})

df_inc = pd.DataFrame(inc_valid + inc_invalid)
print(f'parse_ok=True  : {len(inc_valid):,}')
print(f'parse_ok=False : {len(inc_invalid):,}')
if inc_invalid:
    display(pd.DataFrame(inc_invalid)[['incident_id','severity','parse_ok_reason']].head(3))


parse_ok=True  : 1,245
parse_ok=False : 0


## 4. Pseudonymisation des opérateurs

In [7]:
def badge_hash(name, badge):
    raw = f'{str(name).strip().lower()}|{str(badge).strip().lower()}'
    return hashlib.sha256(raw.encode()).hexdigest()[:16]

df_inc['operator_key'] = df_inc['operator_name'].str.strip().str.lower()
df_inc['badge_hash']   = df_inc.apply(
    lambda r: badge_hash(r['operator_name'], r['operator_badge']), axis=1
)
operators = df_inc[['operator_key', 'badge_hash']].drop_duplicates(subset='operator_key')
print(f'{len(operators)} opérateurs distincts')

with engine.begin() as conn:
    for _, row in operators.iterrows():
        conn.execute(text("""
            INSERT INTO operator (operator_key, badge_hash)
            VALUES (:key, :hash)
            ON CONFLICT (operator_key) DO UPDATE SET badge_hash = EXCLUDED.badge_hash
        """), {'key': row['operator_key'], 'hash': row['badge_hash']})

with engine.connect() as conn:
    n = conn.execute(text('SELECT COUNT(*) FROM operator')).scalar()
print(f'operator : {n} lignes en base')


15 opérateurs distincts
operator : 15 lignes en base


## 5. Création des batches d'ingestion

In [8]:
started_at = datetime.now(timezone.utc)

with engine.begin() as conn:
    batch_tel = conn.execute(text("""
        INSERT INTO ingestion_batch (source_name, source_file, started_at, status)
        VALUES ('telemetry', 'telemetry.csv', :ts, 'running')
        RETURNING ingestion_batch_id
    """), {'ts': started_at}).scalar()

    batch_inc = conn.execute(text("""
        INSERT INTO ingestion_batch (source_name, source_file, started_at, status)
        VALUES ('incidents', 'releves_incidents.csv', :ts, 'running')
        RETURNING ingestion_batch_id
    """), {'ts': started_at}).scalar()

print(f'Batch telemetry : {batch_tel}')
print(f'Batch incidents  : {batch_inc}')


Batch telemetry : 78497474-124a-474d-b847-7111f827697c
Batch incidents  : 90612dcc-9ad5-463a-b16a-4fb2a062f2af


## 6. Ingestion Bronze

In [9]:
df_tel['ingestion_batch_id'] = str(batch_tel)
df_inc['ingestion_batch_id'] = str(batch_inc)

df_tel_typed = df_tel.copy()
for col in ['temperature_c','pressure_bar','voltage_mean_v','rotation_mean_rpm']:
    df_tel_typed[col] = pd.to_numeric(df_tel_typed[col], errors='coerce')
df_tel_typed['pieces_produced'] = pd.to_numeric(df_tel_typed['pieces_produced'], errors='coerce').astype('Int64')

df_inc_typed = df_inc.drop(columns=['operator_key', 'badge_hash']).copy()
for col in ['severity'] + TYPE_COLS:
    df_inc_typed[col] = pd.to_numeric(df_inc_typed[col], errors='coerce').astype('Int64')

with engine.begin() as conn:
    conn.execute(text('TRUNCATE TABLE data_quality_issue'))
    conn.execute(text('TRUNCATE TABLE bronze_telemetry RESTART IDENTITY'))
    conn.execute(text('TRUNCATE TABLE bronze_incidents  RESTART IDENTITY'))

df_tel_typed.to_sql('bronze_telemetry', engine, if_exists='append',
                    index=False, method='multi', chunksize=1000)
df_inc_typed.to_sql('bronze_incidents', engine, if_exists='append',
                    index=False, method='multi', chunksize=1000)

print(f'bronze_telemetry : {len(df_tel_typed):,} lignes')
print(f'bronze_incidents  : {len(df_inc_typed):,} lignes')


bronze_telemetry : 135,626 lignes
bronze_incidents  : 1,245 lignes


## 7. Alimentation de data_quality_issue

In [10]:
dq_rows = []

for _, row in df_tel[~df_tel['parse_ok']].iterrows():
    reason = str(row['parse_ok_reason'])
    rule   = 'DOUBLON_HORAIRE' if 'DOUBLON' in reason else 'PYDANTIC_ERROR'
    dq_rows.append({
        'ingestion_batch_id': str(batch_tel),
        'dataset_name': 'bronze_telemetry',
        'rule_code': rule,
        'severity': 'WARNING' if rule == 'DOUBLON_HORAIRE' else 'ERROR',
        'entity_key': f"{row.get('machine_id', '?')}|{row.get('timestamp', '?')}",
        'details': reason,
    })

for _, row in df_inc[~df_inc['parse_ok']].iterrows():
    dq_rows.append({
        'ingestion_batch_id': str(batch_inc),
        'dataset_name': 'bronze_incidents',
        'rule_code': 'PYDANTIC_ERROR',
        'severity': 'ERROR',
        'entity_key': str(row.get('incident_id', '?')),
        'details': str(row['parse_ok_reason']),
    })

if dq_rows:
    pd.DataFrame(dq_rows).to_sql('data_quality_issue', engine, if_exists='append',
                                 index=False, method='multi', chunksize=500)
    print(f'data_quality_issue : {len(dq_rows):,} lignes')
else:
    print('Aucune violation')


data_quality_issue : 2,686 lignes


## 8. Clôture des batches

In [11]:
finished_at = datetime.now(timezone.utc)

with engine.begin() as conn:
    conn.execute(text("""
        UPDATE ingestion_batch SET
            finished_at = :fin, rows_read = :read,
            rows_loaded = :loaded, rows_rejected = :rejected, status = 'done'
        WHERE ingestion_batch_id = :bid
    """), {'fin': finished_at, 'read': len(df_tel),
           'loaded': int(df_tel['parse_ok'].sum()),
           'rejected': int((~df_tel['parse_ok']).sum()),
           'bid': str(batch_tel)})

    conn.execute(text("""
        UPDATE ingestion_batch SET
            finished_at = :fin, rows_read = :read,
            rows_loaded = :loaded, rows_rejected = :rejected, status = 'done'
        WHERE ingestion_batch_id = :bid
    """), {'fin': finished_at, 'read': len(df_inc),
           'loaded': int(df_inc['parse_ok'].sum()),
           'rejected': int((~df_inc['parse_ok']).sum()),
           'bid': str(batch_inc)})

print('Batches clôturés')


Batches clôturés


## 9. Vérification finale

In [12]:
display(pd.read_sql("""
    SELECT source_name, source_file, rows_read, rows_loaded, rows_rejected, status
    FROM ingestion_batch
    ORDER BY started_at DESC LIMIT 4
""", engine))

display(pd.read_sql("""
    SELECT dataset_name, rule_code, severity, COUNT(*) AS nb
    FROM data_quality_issue
    GROUP BY 1, 2, 3
    ORDER BY 1, 2
""", engine))


,source_name,source_file,rows_read,rows_loaded,rows_rejected,status
0,telemetry,telemetry.csv,135626,132940,2686,done
1,incidents,releves_incidents.csv,1245,1245,0,done
2,telemetry,telemetry.csv,135626,972,134654,done
3,incidents,releves_incidents.csv,1245,1245,0,done


,dataset_name,rule_code,severity,nb
0,bronze_telemetry,DOUBLON_HORAIRE,WARNING,2686
